## Generation Outcome

Outcome: advisories; pipeline: completed.
Usability: no_known_blockers (not full functional verification).
Static QA: passed.
Environment smoke (trusted_smoke_test): not_run.
Generated notebook execution: not_run. Prompt-specific behavior remains unverified.
Blocking findings: 0; advisories: 3.
Candidate (notebook-cells-v1): 38a548a18f88611b22482382f2441a458b719667a6ff8cab23d042919b546d90.
Repair attempts: 0/0; rollback: False.
Checks evidenced: static:canonical_graph_contract, static:canonical_section_order, static:chatbot_notebook_contract, static:domain_architecture_alignment, static:generated_llm_config, static:graph_structure, static:invocation_config, static:json_structure, static:langgraph_topology, static:placeholder_content, static:python_syntax, static:required_import_symbols, static:required_sections, static:state_reducer_semantics, static:tool_reachability, static:undefined_names
No known blockers is not a guarantee that generated code works.

Current QA findings:
- warning / State Reducer Semantics: State reducer advisories detected.
  - Next step: Add reducers for state fields that may receive updates from multiple nodes, or document why writers are mutually exclusive.
- warning / Tool Reachability: Tool reachability advisories detected.
  - Next step: Bind generated tools with model.bind_tools(...) and execute returned tool calls, or pass tools into a documented LangGraph agent/tool node.
  - Next step: For prebuilt agent loops, prefer langchain.agents.create_agent over deprecated create_react_agent.
  - Next step: Label placeholder tools honestly and keep broad external API tools deny-by-default.
- warning / Domain Architecture Alignment: Domain-specific notebook still contains generic or misaligned architecture labels.
  - Next step: Carry request-domain terms into node ids, function names, Mermaid labels, and prose.
  - Next step: Make routing prose match the actual implementation style: Command routes or conditional edges.

# LangGraph Workflow: Build a ticket router for the neighborhood arts ce

Generated by LangGraph Notebook Foundry

**Architecture**: router
**Patterns Used**: router


## Overview

This notebook implements a LangGraph workflow using the **router** pattern.

### Architecture Justification

Architecture forced by request-scoped agent_type override: router.

### Sections

1. Setup
1. State Definition
1. Tools
1. Nodes
1. Build Graph
1. Execution


## Graph Overview

```mermaid
flowchart TD
    START([START]) --> router
    router["router: Handle router requests"]
    billing["billing: Handle billing requests"]
    technical["technical: Handle technical requests"]
    fallback["fallback: Handle fallback requests"]
    router -- "billing" --> technical
    router -- "technical" --> technical
    router -- "fallback" --> fallback
    END([END])
```

### Workflow Schema

```json
{
  "architecture_type": "router",
  "composition_strategy": "router_direct",
  "labels": {
    "title": "Router Workflow"
  },
  "state_schema": {
    "messages": "Conversation messages"
  },
  "nodes": [
    {
      "name": "router",
      "purpose": "Handle router requests",
      "role": null,
      "domain_terms": [],
      "required_tools": []
    },
    {
      "name": "billing",
      "purpose": "Handle billing requests",
      "role": null,
      "domain_terms": [],
      "required_tools": []
    },
    {
      "name": "technical",
      "purpose": "Handle technical requests",
      "role": null,
      "domain_terms": [],
      "required_tools": []
    },
    {
      "name": "fallback",
      "purpose": "Handle fallback requests",
      "role": null,
      "domain_terms": [],
      "required_tools": []
    }
  ],
  "edges": [],
  "conditional_edges": [
    {
      "from": "router",
      "condition": "Select support destination",
      "branches": {
        "billing": "billing",
        "technical": "technical",
        "fallback": "fallback"
      },
      "routing_mechanism": "conditional_edge",
      "guarded_cycle": false
    }
  ],
  "command_routes": [],
  "tool_reachability": [],
  "domain_terms": [
    "support"
  ],
  "compiled_graph_variable": "graph",
  "entry_point": "router",
  "checkpointing": true,
  "terminal_nodes": [
    "billing",
    "technical",
    "fallback"
  ],
  "validation_summary": {
    "errors": [],
    "warnings": []
  }
}
```

## Installation

Install any missing packages needed by this notebook.

### Environment Notes

- The install cell checks for missing packages before invoking pip.
- Generated notebooks assume a Jupyter or Colab-style environment with pip available.

In [ ]:
# Install missing notebook dependencies
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
import subprocess
import sys

PACKAGE_IMPORTS = {
    "langgraph": "langgraph",
    "langchain-core": "langchain_core",
    "langchain-openai>=1.7.2,<2.0.0": "langchain_openai",
    "requests": "requests"
}
REQUIRED_PACKAGES = [
    "langgraph",
    "langchain-core",
    "langchain-openai>=1.7.2,<2.0.0",
    "requests"
]

missing_packages = []
for package in REQUIRED_PACKAGES:
    if find_spec(PACKAGE_IMPORTS[package]) is None:
        missing_packages.append(package)
        continue
    if package.split(">", 1)[0] == "langchain-openai":
        try:
            installed_version = version("langchain-openai")
        except PackageNotFoundError:
            missing_packages.append(package)
        else:
            if Version(installed_version) < Version("1.7.2"):
                missing_packages.append(package)

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("All required packages are already available.")

## Configuration

Set up API keys and configuration:

In [ ]:
import os
from urllib.parse import urlsplit
import ipaddress

from langchain_openai import ChatOpenAI

# Configuration
MODEL = "gpt-6-luna"
TEMPERATURE = 0.0
MAX_ITERATIONS = 10
MAX_TOKENS = None
API_BASE = "https://api.openai.com/v1"
CUSTOM_LLM_PROVIDER = False
LLM_PROVIDER_ID = "openai"
# Custom providers need LNF_NOTEBOOK_API_BASE and their own LNF_NOTEBOOK_LLM_API_KEY.
# Explicit loopback-only no-auth: LNF_NOTEBOOK_LLM_NO_AUTH=true.
THREAD_ID = "lnf-demo-thread"
SHOW_UPDATES = False
RUN_INTERACTIVE_LOOP = False
RUN_DEMO_TURNS = False
CHARACTER_GENDER = None  # Set to 'male' or 'female' to skip the first-turn prompt.
ANACHRONISM_TERMS = (
    "smartphone",
    "internet",
    "television",
    "movie",
    "star wars",
    "computer",
    "airplane",
    "radio",
    "electricity",
)

# Credentials
# Prefer environment variables over hardcoded secrets in notebooks.
def _load_env_var(name: str) -> str:
    """Load a credential from env, Colab secrets, or an interactive prompt."""
    value = os.environ.get(name, '')
    if value:
        return value
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name) or ''
    except Exception:
        value = ''
    if value:
        os.environ[name] = value
        return value
    try:
        from getpass import getpass
        value = getpass(f'Enter {name}: ')
    except (EOFError, KeyboardInterrupt):
        value = ''
    if value:
        os.environ[name] = value
    return value

OPENAI_API_KEY = _load_env_var("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("OPENAI_API_KEY is not set. Configure it before running live model cells.")


def make_llm(
    *,
    model: str = MODEL,
    temperature: float = TEMPERATURE,
    max_tokens: int | None = MAX_TOKENS,
):
    """Construct a ChatOpenAI client from notebook-level settings."""
    kwargs: dict[str, object] = {
        "model": model,
    }
    if model == 'gpt-6-luna' and not CUSTOM_LLM_PROVIDER and API_BASE in ('https://api.openai.com/v1', 'https://api.openai.com/v1/'):
        kwargs.update(
            use_responses_api=True,
            reasoning={"effort": "low"},
            output_version="responses/v1",
        )
    else:
        kwargs["temperature"] = temperature
    if not API_BASE:
        raise ValueError("Set LNF_NOTEBOOK_API_BASE for this provider.")
    parsed = urlsplit(API_BASE)
    if parsed.username or parsed.password or parsed.query or parsed.fragment or any(c in API_BASE for c in "\\%@?#"):
        raise ValueError("Use a credential-free endpoint URL.")
    try:
        loopback = ipaddress.ip_address(parsed.hostname or '').is_loopback
    except ValueError:
        loopback = parsed.hostname == 'localhost'
    if not parsed.hostname or parsed.scheme not in {'https', 'http'} or (parsed.scheme == 'http' and not loopback):
        raise ValueError("Remote endpoints require HTTPS.")
    key_env = 'LNF_NOTEBOOK_LLM_API_KEY' if CUSTOM_LLM_PROVIDER else 'OPENAI_API_KEY'
    key = os.environ.get(key_env, '')
    no_auth = CUSTOM_LLM_PROVIDER and os.environ.get('LNF_NOTEBOOK_LLM_NO_AUTH', '').lower() == 'true'
    if no_auth:
        if key or not loopback:
            raise ValueError("No-auth requires loopback and no custom key.")
        key = 'lnf-local-no-auth'
    if not key:
        raise ValueError(f"Set {key_env} before live model calls.")
    kwargs["base_url"] = API_BASE
    kwargs["api_key"] = key
    if max_tokens is not None:
        kwargs["max_tokens"] = max_tokens
    return ChatOpenAI(**kwargs)

## State Schema

Define the workflow state:

In [ ]:
import operator
from typing import Annotated, Dict, List
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages


def merge_dicts(left: Dict[str, str], right: Dict[str, str]) -> Dict[str, str]:
    """Reducer used to merge route outputs into a single state field."""
    merged = dict(left or {})
    merged.update(right or {})
    return merged


class WorkflowState(TypedDict, total=False):
    """State schema for a router workflow."""

    messages: Annotated[List[BaseMessage], add_messages]
    route: str
    route_reasoning: str
    route_history: Annotated[List[str], operator.add]
    results: Annotated[Dict[str, str], merge_dicts]
    final_output: str


## Tool Planning Notes

Review these advisories before relying on the generated tool plan as-is.

- Fallback used: Tool planning fell back to heuristic inference because the model returned no usable canonical tools after validation and environment filtering.

- Additional warnings:
  - Tool planning fell back to heuristic inference because the model returned no usable canonical tools after validation and environment filtering.

## Tools

Define tools used in the workflow:

In [ ]:
# Tool: HTTP Client
# Purpose: Heuristically inferred from workflow nodes: router, billing, technical, fallback
# Category: api

from langchain_core.tools import tool

@tool
def calculator(a: int, b: int) -> int:
    """Add two integers deterministically without external services."""
    return a + b

In [ ]:
from langgraph.prebuilt import ToolNode

TOOLS = [calculator]
TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}
tool_node = ToolNode(TOOLS, handle_tool_errors=True)

In [ ]:
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


class RouteDecision(BaseModel):
    """Typed routing decision returned by the classifier."""

    route: Literal["billing", "technical", "fallback"] = Field(
        description="Which specialist should handle the request."
    )
    reasoning: str = Field(
        description="Why this route is the best match for the request."
    )


def router_node(state: WorkflowState, window_size: int = 5) -> WorkflowState:
    """Routes requests to appropriate specialist based on input classification.
    
    Analyzes the user's message and determines which specialized agent
    should handle the request based on content and intent.
    """
    messages = state["messages"]
    last_message = messages[-1].content if messages else ""
    window_size = max(window_size, 1)
    recent_messages = messages[-window_size:] if messages else []
    conversation_history = "\n".join(
        f"{getattr(message, 'type', message.__class__.__name__)}: {message.content}"
        for message in recent_messages
    ) or "No prior conversation available."
    
    # Initialize LLM with structured output
    llm = make_llm()
    structured_llm = llm.with_structured_output(RouteDecision)
    
    # Classification prompt
    system_prompt = SystemMessage(content="""You are a routing classifier.
Analyze the recent conversation and select the most appropriate route.
Resolve coreferences such as "it", "that", and "the one" using the conversation history.
Use the fallback route for greetings, generic chat, unsupported requests, missing-tool requests, or anything that does not clearly belong to a specialist.
Respond using the provided RouteDecision schema.""")

    classification_prompt = f"""Analyze the following conversation and determine which route should handle the user's latest request.

Available routes:
- billing: Handle billing-specific requests.
- technical: Handle technical-specific requests.
- fallback: Handle unclear, conversational, unsupported, or out-of-domain requests.

Recent conversation (last {window_size} messages):
{conversation_history}

Latest user request: {last_message}

Select the most appropriate route and explain your reasoning."""
    
    # Get classification
    decision = structured_llm.invoke([system_prompt, HumanMessage(content=classification_prompt)])
    
    return {
        "route": decision.route,
        "route_reasoning": decision.reasoning,
        "route_history": [decision.route],
        "messages": [AIMessage(content=f"Routing to: {decision.route} ({decision.reasoning})")],
    }

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage
from langchain_openai import ChatOpenAI


def billing_node(state: WorkflowState) -> dict:
    """Execute the billing specialist."""
    messages = state.get("messages", [])
    llm = make_llm()

    response = llm.invoke([
        SystemMessage(
            content="""You are the billing specialist.

Responsibility:
Handle billing requests
"""
        ),
        *messages,
    ])

    return {
        "results": {"billing": response.text},
        "final_output": response.text,
        "messages": [AIMessage(content=f"billing specialist completed the task.")],
    }

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage
from langchain_openai import ChatOpenAI


def technical_node(state: WorkflowState) -> dict:
    """Execute the technical specialist."""
    messages = state.get("messages", [])
    llm = make_llm()

    response = llm.invoke([
        SystemMessage(
            content="""You are the technical specialist.

Responsibility:
Handle technical requests
"""
        ),
        *messages,
    ])

    return {
        "results": {"technical": response.text},
        "final_output": response.text,
        "messages": [AIMessage(content=f"technical specialist completed the task.")],
    }

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage
from langchain_openai import ChatOpenAI


def fallback_node(state: WorkflowState) -> dict:
    """Execute the fallback specialist."""
    messages = state.get("messages", [])
    llm = make_llm()

    response = llm.invoke([
        SystemMessage(
            content="""You are the fallback specialist.

Responsibility:
Handle fallback requests
"""
        ),
        *messages,
    ])

    return {
        "results": {"fallback": response.text},
        "final_output": response.text,
        "messages": [AIMessage(content=f"fallback specialist completed the task.")],
    }

## Build Graph

Define the LangGraph workflow structure.

In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver


CANONICAL_GRAPH_SPEC = {'architecture_type': 'router', 'composition_strategy': 'router_direct', 'labels': {'title': 'Router Workflow'}, 'state_schema': {'messages': 'Conversation messages'}, 'nodes': [{'name': 'router', 'purpose': 'Handle router requests', 'role': None, 'domain_terms': [], 'required_tools': []}, {'name': 'billing', 'purpose': 'Handle billing requests', 'role': None, 'domain_terms': [], 'required_tools': []}, {'name': 'technical', 'purpose': 'Handle technical requests', 'role': None, 'domain_terms': [], 'required_tools': []}, {'name': 'fallback', 'purpose': 'Handle fallback requests', 'role': None, 'domain_terms': [], 'required_tools': []}], 'edges': [], 'conditional_edges': [{'from': 'router', 'condition': 'Select support destination', 'branches': {'billing': 'technical', 'technical': 'technical', 'fallback': 'fallback'}, 'routing_mechanism': 'conditional_edge', 'guarded_cycle': False}], 'command_routes': [], 'tool_reachability': [], 'domain_terms': ['support'], 'compiled_graph_variable': 'graph', 'entry_point': 'router', 'checkpointing': True, 'terminal_nodes': ['billing', 'technical', 'fallback'], 'validation_summary': {'errors': [], 'warnings': []}}


# Create graph from canonical graph/spec metadata.
workflow = StateGraph(WorkflowState)
checkpointer = InMemorySaver()


# Add nodes.
workflow.add_node("router", router_node)
workflow.add_node("billing", billing_node)
workflow.add_node("technical", technical_node)
workflow.add_node("fallback", fallback_node)

# Connect entry point and canonical direct edges.
workflow.add_edge(START, "router")
workflow.add_edge("billing", END)
workflow.add_edge("technical", END)
workflow.add_edge("fallback", END)

# Add canonical conditional and Command-route destinations.
def _route_from_router_0(state: WorkflowState):
    path_map = {"billing": "technical", "technical": "technical", "fallback": "fallback"}
    for field_name in ("next_agent", "route", "decision", "status"):
        value = state.get(field_name)
        if value in path_map:
            return value
    return "billing"

workflow.add_conditional_edges("router", _route_from_router_0, {"billing": "technical", "technical": "technical", "fallback": "fallback"})

# Compile graph with the canonical variable name.
graph = workflow.compile(checkpointer=checkpointer)

## Execution

Run the workflow:

In [ ]:
from langchain_core.messages import HumanMessage

# Execute the workflow with a durable thread and bounded graph recursion.
config = {"configurable": {"thread_id": "lnf-demo-thread"}, "recursion_limit": 25}
initial_state: WorkflowState = {
    "messages": [HumanMessage(content="Route this request to the best specialist and produce a concise answer.")],
    "route": "",
    "results": {},
    "final_output": "",
}

print("Streaming workflow state:")
for step in graph.stream(initial_state, config, stream_mode="values"):
    print(step)

final_state = graph.get_state(config).values
print(final_state)

## Export Results
Persist outputs for downstream use.

In [ ]:
import json
from pathlib import Path

if "graph" in globals():
    compiled_graph = globals()["graph"]
else:
    compiled_graph = globals().get("app")
if compiled_graph is None:
    raise NameError("`graph` is not defined. Run the 'Build Graph' cell before exporting results.")

config = {"configurable": {"thread_id": "lnf-demo-thread"}, "recursion_limit": 25}
final_state_snapshot = globals().get("final_state")
if final_state_snapshot is not None:
    output_data = final_state_snapshot
else:
    output_data = compiled_graph.get_state(config).values
output_path = Path("graph_results.json")
with output_path.open("w", encoding="utf-8") as handle:
    json.dump(output_data, handle, indent=2, default=str)

print(f"Saved results to {output_path.resolve()}")

## Troubleshooting
- Restart runtime if imports fail in Colab.
- Confirm `OPENAI_API_KEY` is set before running graph cells.
- Ensure pip installs complete before executing later cells.
- Review output JSON for unexpected schema mismatches.